# Seasonal Agriculture Performance Analysis

This project analyzes agricultural performance data across three farming seasons — Kharif, Rabi, and Zaid — to identify meaningful patterns, trends, and differences in yield, profitability, resource usage, and environmental conditions. The dataset contains 4,000 farm records across 8 Indian states with 28 attributes covering farming practices, environmental factors, and economic outcomes.

The goal is to understand how agricultural performance varies by season, uncover the key drivers behind these differences, and provide evidence-based recommendations to support better seasonal agricultural planning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("/content/seasonal_agriculture_performance_dataset.csv")
df.head()

In [ ]:
print("Rows and Columns:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
df.describe()

In [ ]:
for col in ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']:
    df[col] = df.groupby('Season')[col].transform(lambda x: x.fillna(x.median()))

print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
print("Seasons:", df['Season'].unique())
print("Crops:", df['Crop'].unique())
print("Irrigation Methods:", df['Irrigation_Method'].unique())
print("States:", df['State'].unique())

Key Analytical Questions:

How does yield differ across Kharif, Rabi, and Zaid seasons?
Which season generates the highest profit and revenue?
How does water usage and water efficiency vary by season?
Is there a relationship between rainfall and yield across seasons?
Which season carries the highest disease/pest risk, and does it affect profit?
Does irrigation method affect yield differently across seasons?




In [ ]:
season_summary = df.groupby('Season')[[
    'Yield_Tonnes_Ha', 'Production_Tonnes', 'Profit_INR', 'Revenue_INR',
    'Total_Cost_INR', 'Water_Used_m3', 'Water_Efficiency_t_per_1000m3',
    'Rainfall_mm', 'Disease_Pest_Risk_pct'
]].mean().round(2)

season_summary

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x='Season', y='Profit_INR', data=df)
plt.title('Profit Distribution by Season')
plt.axhline(0, color='red', linestyle='--')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x='Season', y='Yield_Tonnes_Ha', data=df)
plt.title('Yield Distribution by Season')
plt.show()

In [ ]:
season_summary_full = df.groupby('Season')[[
    'Disease_Pest_Risk_pct', 'Water_Efficiency_t_per_1000m3',
    'Seed_Quality_Score', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha'
]].mean().round(2)

season_summary_full

In [ ]:
df.groupby('Season')[['Total_Cost_INR', 'Revenue_INR', 'Market_Price_INR_Tonne']].mean().round(2)

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df[df['Season']=='Zaid'].groupby('Crop')[['Yield_Tonnes_Ha','Profit_INR']].mean().round(2)

In [ ]:
df[df['Season']=='Zaid']['Crop'].value_counts()

In [ ]:
df[df['Season']=='Kharif'].groupby('Crop')[['Yield_Tonnes_Ha','Profit_INR']].mean().round(2)
df[df['Season']=='Rabi'].groupby('Crop')[['Yield_Tonnes_Ha','Profit_INR']].mean().round(2)

In [ ]:
from scipy import stats
kharif_p = df[df['Season']=='Kharif']['Profit_INR']
rabi_p = df[df['Season']=='Rabi']['Profit_INR']
zaid_p = df[df['Season']=='Zaid']['Profit_INR']

f_stat, p_val = stats.f_oneway(kharif_p, rabi_p, zaid_p)
print("F-statistic:", f_stat, "P-value:", p_val)

In [ ]:
kharif_y = df[df['Season']=='Kharif']['Yield_Tonnes_Ha']
rabi_y = df[df['Season']=='Rabi']['Yield_Tonnes_Ha']
zaid_y = df[df['Season']=='Zaid']['Yield_Tonnes_Ha']

f_stat_y, p_val_y = stats.f_oneway(kharif_y, rabi_y, zaid_y)
print("Yield ANOVA — F-statistic:", f_stat_y, "P-value:", p_val_y)

In [ ]:
plt.figure(figsize=(9,5))
zaid_data = df[df['Season']=='Zaid']
sns.barplot(x='Crop', y='Profit_INR', data=zaid_data, estimator=np.mean, errorbar=None)
plt.title('Average Profit by Crop — Zaid Season')
plt.axhline(0, color='red', linestyle='--')
plt.xticks(rotation=45)
plt.show()

In [ ]:
kharif_y = df[df['Season']=='Kharif']['Yield_Tonnes_Ha']
rabi_y = df[df['Season']=='Rabi']['Yield_Tonnes_Ha']
zaid_y = df[df['Season']=='Zaid']['Yield_Tonnes_Ha']

f_stat_y, p_val_y = stats.f_oneway(kharif_y, rabi_y, zaid_y)
print("Yield ANOVA — F-statistic:", f_stat_y, "P-value:", p_val_y)

In [ ]:
kharif_p = df[df['Season']=='Kharif']['Profit_INR']
rabi_p = df[df['Season']=='Rabi']['Profit_INR']
zaid_p = df[df['Season']=='Zaid']['Profit_INR']

f_stat_p, p_val_p = stats.f_oneway(kharif_p, rabi_p, zaid_p)
print("Profit ANOVA — F-statistic:", f_stat_p, "P-value:", p_val_p)

## Key Findings

1. Kharif is the most profitable season, with the highest average profit (₹1,78,915) and best cost-to-revenue balance.

2. Yield does not vary significantly by season (ANOVA: F=1.55, p=0.212) — seasonal choice does not meaningfully affect how much a farm produces per hectare.

3. Profit varies significantly by season (ANOVA: F=34.29, p≈1.71e-15) — despite similar yields, seasons differ hugely in profitability, driven by cost structure and crop-market fit rather than raw production.

4. Zaid season runs at a net average loss (-₹24,805), not due to disease/pest risk (which is actually lowest in Zaid at 38.22%) but due to poor crop-season fit and the lowest water efficiency (4.41 t/1000m³) of all seasons.

5. The Zaid loss is crop-specific. Sugarcane (+₹5.8L) and Chilli (+₹4.5L) are highly profitable in Zaid, while Maize, Wheat, Rice, Pulses, Cotton, and Groundnut all post losses (-₹54K to -₹227K), suggesting these crops are being grown outside their optimal season.

6. Yield strongly correlates with Revenue (0.91) and Production (0.88), confirming internal consistency of the dataset.

## Recommendations

1. Discourage off-season cultivation of Maize, Wheat, Rice, Pulses, and Cotton during Zaid — these crops consistently post losses in this season.

2. Promote Sugarcane and Chilli cultivation in Zaid — the only crops showing strong profitability that season.

3. Improve water-use efficiency in Zaid through drip irrigation, since it has the lowest water efficiency despite high consumption.

4. Prioritize Kharif-season investment and support, since it consistently delivers the best profit outcomes.

5. Further investigate why non-suited crops are still grown in Zaid — may point to farmer awareness gaps or market/seed access issues.